In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

import matplotlib.pyplot as plt


# =============================================================================
# Task 5 — Dominant SMOD L1 class per ADM2 (by cell counts)
# Outputs (all prefixed with task5_):
#   tables/  : CSVs
#   outputs/ : JSON + (optional) GPKG
#   figures/ : PNGs
# =============================================================================

# ----------------------------
# Project structure
# ----------------------------
ROOT = Path(r"C:\Users\am636\copperbelt_worldpop_smod")

DATA = ROOT / "data_raw"
OUT = ROOT / "outputs"
TABDIR = ROOT / "tables"
FIGDIR = ROOT / "figures"

for d in (DATA, OUT, TABDIR, FIGDIR):
    d.mkdir(parents=True, exist_ok=True)

PREFIX = "task5_"

boundary_path = DATA / "ZMB_adm2_gadm41_Copperbelt.shp"
smod_l1_path = OUT / "task4_smod_L1_from_L2_54009_1km_INT16.tif"

missing = [p for p in (boundary_path, smod_l1_path) if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required input file(s):\n" + "\n".join(str(p) for p in missing))

# Options
ALL_TOUCHED = False
NAME_FIELD = "NAME_2"
NODATA_L1 = -200

run_time_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")


# ----------------------------
# Output paths
# ----------------------------
out_csv = TABDIR / f"{PREFIX}smodL1_cellcounts_by_adm2.csv"
out_sum = TABDIR / f"{PREFIX}smodL1_dominant_summary.csv"
out_json = OUT / f"{PREFIX}smodL1_dominant_report.json"
out_map = FIGDIR / f"{PREFIX}smodL1_dominant_map.png"
out_bar = FIGDIR / f"{PREFIX}smodL1_cellcount_stackedbar.png"
out_gpkg = OUT / f"{PREFIX}adm2_with_dominant.gpkg"


# ----------------------------
# Helpers
# ----------------------------
def rasterised_class_counts(gdf_proj: gpd.GeoDataFrame, raster_path: Path, nodata: int, all_touched: bool) -> pd.DataFrame:
    """
    Per-polygon categorical counts of raster values (excluding nodata).
    Returns a dataframe with columns: idx, value, cell_count.
    """
    from rasterio.features import rasterize

    with rasterio.open(raster_path) as src:
        arr = src.read(1)
        transform = src.transform
        height, width = arr.shape

    records = []
    for i, geom in enumerate(gdf_proj.geometry):
        if geom is None or geom.is_empty:
            continue

        mask = rasterize(
            [(geom, 1)],
            out_shape=(height, width),
            transform=transform,
            fill=0,
            all_touched=all_touched,
            dtype="uint8",
        )

        vals = arr[mask == 1]
        vals = vals[vals != nodata]

        if vals.size == 0:
            continue

        u, c = np.unique(vals, return_counts=True)
        for v, n in zip(u.tolist(), c.tolist()):
            records.append({"row_index": i, "value": int(v), "cell_count": int(n)})

    return pd.DataFrame(records)


def dominant_from_counts(c1: int, c2: int, c3: int):
    total = c1 + c2 + c3
    if total == 0:
        return np.nan, "NO_DATA", False

    counts = {1: c1, 2: c2, 3: c3}
    maxv = max(counts.values())
    winners = [k for k, v in counts.items() if v == maxv]

    if len(winners) == 1:
        code = winners[0]
        label = {1: "Rural", 2: "Urban cluster", 3: "Urban centre"}[code]
        return code, label, False

    return np.nan, "TIE", True


# ----------------------------
# 1) Read inputs + reproject boundaries to SMOD CRS
# ----------------------------
gdf = gpd.read_file(boundary_path)

if gdf.crs is None:
    raise ValueError("Boundary CRS missing.")

if NAME_FIELD not in gdf.columns:
    raise ValueError(f"{NAME_FIELD} not found in boundary attributes. Available: {list(gdf.columns)}")

with rasterio.open(smod_l1_path) as src:
    smod_crs = src.crs
    smod_dtype = src.dtypes[0]
    smod_nodata = src.nodata
    smod_shape = (src.height, src.width)

if smod_nodata is None:
    smod_nodata = NODATA_L1

gdf_smod = gdf.to_crs(smod_crs)

print("Inputs:")
print(" - ADM2:", boundary_path.name, "| features:", len(gdf_smod), "| CRS:", gdf_smod.crs)
print(" - SMOD L1:", smod_l1_path.name, "| CRS:", smod_crs, "| dtype:", smod_dtype, "| nodata:", smod_nodata, "| shape:", smod_shape)
print(" - all_touched:", ALL_TOUCHED)
print()


# ----------------------------
# 2) Per-ADM2 cell counts per class
# ----------------------------
counts_long = rasterised_class_counts(gdf_smod, smod_l1_path, int(smod_nodata), ALL_TOUCHED)

if counts_long.empty:
    df = pd.DataFrame({
        "district": gdf_smod[NAME_FIELD].astype(str).tolist(),
        "cells_rural": 0,
        "cells_urban_cluster": 0,
        "cells_urban_centre": 0,
        "cells_total": 0,
        "prop_rural": np.nan,
        "prop_urban_cluster": np.nan,
        "prop_urban_centre": np.nan,
        "dominant_class": "NO_DATA",
        "dominant_code": np.nan,
        "tie_for_dominant": False,
    })
else:
    wide = (counts_long
            .pivot_table(index="row_index", columns="value", values="cell_count", aggfunc="sum", fill_value=0)
            .reset_index())

    for code in [1, 2, 3]:
        if code not in wide.columns:
            wide[code] = 0

    rows = []
    for i in range(len(gdf_smod)):
        nm = str(gdf_smod.iloc[i][NAME_FIELD])

        row = wide[wide["row_index"] == i]
        c1 = int(row[1].iloc[0]) if not row.empty else 0
        c2 = int(row[2].iloc[0]) if not row.empty else 0
        c3 = int(row[3].iloc[0]) if not row.empty else 0

        total = c1 + c2 + c3
        pr = (c1 / total) if total > 0 else np.nan
        pu = (c2 / total) if total > 0 else np.nan
        pc = (c3 / total) if total > 0 else np.nan

        dom_code, dom_label, tie = dominant_from_counts(c1, c2, c3)

        rows.append({
            "district": nm,
            "cells_rural": c1,
            "cells_urban_cluster": c2,
            "cells_urban_centre": c3,
            "cells_total": total,
            "prop_rural": pr,
            "prop_urban_cluster": pu,
            "prop_urban_centre": pc,
            "dominant_class": dom_label,
            "dominant_code": dom_code,
            "tie_for_dominant": tie,
        })

    df = pd.DataFrame(rows)

df = df.sort_values(["cells_total", "district"], ascending=[False, True]).reset_index(drop=True)
df.to_csv(out_csv, index=False)
print("Saved:", out_csv)


# ----------------------------
# 3) Dominant-class summary
# ----------------------------
summary = (df["dominant_class"]
           .value_counts(dropna=False)
           .rename_axis("dominant_class")
           .reset_index(name="n_adm2"))

summary["share"] = summary["n_adm2"] / len(df)
summary.to_csv(out_sum, index=False)
print("Saved:", out_sum)

n_urban_centre_dom = int((df["dominant_class"] == "Urban centre").sum())
n_rural_dom = int((df["dominant_class"] == "Rural").sum())

print("\nTask 5 counts:")
print(" - Urban centre dominant:", n_urban_centre_dom)
print(" - Rural dominant:", n_rural_dom)
print()


# ----------------------------
# 4) Map of dominant class
# ----------------------------
gdf_out = gdf_smod.copy()
gdf_out = gdf_out.merge(df[["district", "dominant_class"]], left_on=NAME_FIELD, right_on="district", how="left")

fig, ax = plt.subplots(figsize=(8, 7))
gdf_out.plot(column="dominant_class", ax=ax, legend=True, linewidth=0.8, edgecolor="black")
ax.set_title("Dominant SMOD L1 class by ADM2 (cell-count proxy)")
ax.set_axis_off()
plt.tight_layout()
plt.savefig(out_map, dpi=300, bbox_inches="tight")
plt.close(fig)

print("Saved:", out_map)


# ----------------------------
# 5) Stacked bar of cell counts per ADM2
# ----------------------------
plot_df = df.set_index("district")[["cells_rural", "cells_urban_cluster", "cells_urban_centre"]]

fig, ax = plt.subplots(figsize=(10, 5))
plot_df.plot(kind="bar", stacked=True, ax=ax)
ax.set_ylabel("Number of SMOD L1 grid cells")
ax.set_title("ADM2 settlement composition by SMOD L1 (cell counts)")
ax.tick_params(axis="x", labelrotation=60)
plt.tight_layout()
plt.savefig(out_bar, dpi=300, bbox_inches="tight")
plt.close(fig)

print("Saved:", out_bar)


# ----------------------------
# 6) Optional GeoPackage
# ----------------------------
try:
    gdf_out.to_file(out_gpkg, layer="adm2_dominant_smod_l1", driver="GPKG")
    wrote_gpkg = True
    print("Saved:", out_gpkg)
except Exception as e:
    wrote_gpkg = False
    print("GeoPackage write skipped:", e)


# ----------------------------
# 7) JSON report
# ----------------------------
report = {
    "run_time_utc": run_time_utc,
    "inputs": {
        "boundary_path": str(boundary_path),
        "smod_l1_path": str(smod_l1_path),
        "boundary_crs_original": str(gdf.crs),
        "boundary_crs_used": str(gdf_smod.crs),
        "smod_crs": str(smod_crs),
        "smod_dtype": str(smod_dtype),
        "smod_nodata": int(smod_nodata),
        "all_touched": ALL_TOUCHED,
        "name_field": NAME_FIELD,
        "method": "rasterio.features.rasterize (per-feature mask, categorical counts)",
    },
    "outputs": {
        "per_adm2_csv": str(out_csv),
        "dominant_summary_csv": str(out_sum),
        "dominant_map_png": str(out_map),
        "stacked_bar_png": str(out_bar),
        "adm2_gpkg": str(out_gpkg) if wrote_gpkg else None,
    },
    "results": {
        "n_adm2": int(len(df)),
        "dominant_counts": summary.to_dict(orient="records"),
        "answer_counts": {
            "urban_centre_dominant": n_urban_centre_dom,
            "rural_dominant": n_rural_dom,
        },
    },
    "rules": {
        "dominance": "max(cell_counts) over {1,2,3}; ties labelled TIE; zero cells labelled NO_DATA",
        "nodata_excluded": int(smod_nodata),
    },
}

with open(out_json, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("Saved:", out_json)


Inputs:
 - ADM2: ZMB_adm2_gadm41_Copperbelt.shp | features: 12 | CRS: PROJCS["World_Mollweide",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Mollweide"],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
 - SMOD L1: 04_smod_L1_from_L2_54009_1km_INT16.tif | CRS: ESRI:54009 | dtype: int16 | nodata: -200.0 | shape: (1000, 1000)
 - all_touched: False

Saved: C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\05_smodL1_cellcounts_by_adm2.csv
Saved: C:\Users\am636\Desktop\Job\applied\Shortlisted\Specialist Technician_Demographic Data\Interview_package2\in_phyton\outputs\05_smodL1_dominant_summary.csv

Task 5 answer (boundary as provided):
 - ADM2 domin